In [1]:
import sys
sys.path.insert(0, '../src')   # tells the notebook where to find aura

from aura.tools.profiler import profile_dataset, print_profile

profile = profile_dataset("../data/samples/titanic.csv")
print_profile(profile)


SOURCE : ../data/samples/titanic.csv
TYPE   : CSV
TABLES : 1
COLUMNS: 12
ROWS   : 891

  TABLE: titanic (891 rows × 12 cols)
  CANDIDATE KEYS: ['PassengerId', 'Name']
    PassengerId                    numeric          distinct=891    nulls=0.0%
    Survived                       numeric          distinct=2      nulls=0.0%
    Pclass                         numeric          distinct=3      nulls=0.0%
    Name                           id_or_code       distinct=891    nulls=0.0%
    Sex                            categorical      distinct=2      nulls=0.0%
    Age                            numeric          distinct=88     nulls=19.87%
    SibSp                          numeric          distinct=7      nulls=0.0%
    Parch                          numeric          distinct=7      nulls=0.0%
    Ticket                         categorical      distinct=681    nulls=0.0%
    Fare                           numeric          distinct=248    nulls=0.0%
    Cabin                          categ

In [2]:
from aura.agents.schema_agent import run_schema_agent

result = run_schema_agent("../data/samples/titanic.csv")
enrichment = result["enrichment"]

print("DATASET SUMMARY")
print("=" * 60)
print(enrichment["dataset_summary"])

print("\nCOLUMN DESCRIPTIONS")
print("=" * 60)
for col, desc in enrichment["column_descriptions"].items():
    print(f"  {col:<15} {desc}")

print("\nDATA QUALITY ISSUES")
print("=" * 60)
for issue in enrichment["data_quality_issues"]:
    print(f"  [{issue['severity'].upper()}] {issue['column']}: {issue['issue']}")

print("\nTOP 5 ANALYTICAL QUESTIONS THIS DATASET CAN ANSWER")
print("=" * 60)
for i, q in enumerate(enrichment["analytical_questions"], 1):
    print(f"  {i}. {q}")

print(f"\nRECOMMENDED TARGET VARIABLE : {enrichment.get('recommended_target_variable')}")
print(f"OVERALL DATA QUALITY SCORE  : {enrichment.get('overall_quality_score')}/100")
print(f"\nCOST THIS RUN: ${result['cost']['total_usd']:.6f}")

Running deterministic profiler...
Sending profile to Gemini (12 columns)...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 41.788253728s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

In [3]:
import importlib
import aura.core.llm as llm_module
import aura.agents.schema_agent as schema_module
import aura.agents.engineer_agent as eng_module

importlib.reload(llm_module)
importlib.reload(schema_module)
importlib.reload(eng_module)

from aura.agents.schema_agent import run_schema_agent
from aura.agents.engineer_agent import run_engineer_agent

# Temporarily patch to use gemini-2.0-flash which is more available
import aura.config as config
config.GEMINI_MODEL = "gemini-2.0-flash"

# Step 1: Profile the database
print("=== PROFILING DATABASE ===")
schema_result = run_schema_agent("../data/samples/northwind.db")

# Step 2: Ask a business question
question = "What are the top 5 products by total revenue?"
print(f"\n=== ENGINEER AGENT ===")
print(f"Question: {question}")

eng_result = run_engineer_agent(
    question=question,
    db_path="../data/samples/northwind.db",
    schema_profile=schema_result["profile"],
)

# Step 3: Show results
print("\n=== RESULTS ===")
if eng_result["success"]:
    print(f"Final SQL:\n{eng_result['final_sql']}\n")
    print(eng_result["dataframe"].to_string(index=False))
else:
    print("All attempts failed. Attempt log:")
    for a in eng_result["attempts"]:
        print(f"  Attempt {a['attempt']}: {a['error']}")

print(f"\nAttempts needed  : {eng_result['total_attempts']}")
print(f"Total cost so far: ${eng_result['cost']['total_usd']:.6f}")

=== PROFILING DATABASE ===
Running deterministic profiler...
Sending profile to Gemini (88 columns)...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 41.961884107s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},